# EdgeMason: Free Cloud Edge GPU Benchmark (NVIDIA T4)
This notebook benchmarks all 5 CNN models (**MobileNetV2, ResNet-18, ResNet-50, ShuffleNetV2, SqueezeNet**) across **FP32, FP16, and INT8** on an NVIDIA T4 GPU with Tensor Cores.

### Instructions:
1. Go to Menu: **Runtime** -> **Change runtime type** -> Ensure **T4 GPU** is selected.
2. Go to Menu: **Runtime** -> **Run all** (or press Ctrl+F9).
3. At the end, the generated dataset `benchmark_gpu_t4.csv` will be automatically downloaded to your computer!

In [ ]:
# 1. Environment Setup
!nvidia-smi
!pip install --upgrade pip
!pip install onnxruntime-gpu onnx torchvision pandas numpy psutil

In [ ]:
# 2. Export 5 Canonical CNN Models to ONNX
import os, torch, torchvision.models as models

os.makedirs('models/samples', exist_ok=True)
os.makedirs('reports/data', exist_ok=True)

model_map = {
    'mobilenetv2.onnx': models.mobilenet_v2(weights=None),
    'resnet18.onnx': models.resnet18(weights=None),
    'resnet50.onnx': models.resnet50(weights=None),
    'shufflenetv2.onnx': models.shufflenet_v2_x1_0(weights=None),
    'squeezenet.onnx': models.squeezenet1_1(weights=None)
}

dummy_input = torch.randn(1, 3, 224, 224)

print('Exporting models to ONNX...')
for name, model in model_map.items():
    model.eval()
    path = os.path.join('models', name)
    torch.onnx.export(
        model,
        dummy_input,
        path,
        input_names=['input'],
        output_names=['output'],
        opset_version=13,
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
    )
    print(f'Exported {name} ({os.path.getsize(path)/1024/1024:.2f} MB)')

# Create calibration sample
import numpy as np
calib_data = np.random.randn(50, 3, 224, 224).astype(np.float32)
np.savez('models/samples/calibration_50.npz', data=calib_data)
print('Calibration dataset generated.')

In [ ]:
# 3. Benchmark Engine across FP32, FP16, and INT8 on CUDA / Tensor Cores
import time, onnxruntime as ort, pandas as pd
from onnxruntime.quantization import quantize_dynamic, quantize_static, CalibrationDataReader, QuantType

class NpzDataReader(CalibrationDataReader):
    def __init__(self, npz_path, input_name='input'):
        self.data = np.load(npz_path)['data']
        self.input_name = input_name
        self.enum_data = iter([{self.input_name: self.data[i:i+1]} for i in range(len(self.data))])
    def get_next(self):
        return next(self.enum_data, None)

results = []
models_list = ['mobilenetv2.onnx', 'resnet18.onnx', 'resnet50.onnx', 'shufflenetv2.onnx', 'squeezenet.onnx']

for model_name in models_list:
    model_path = os.path.join('models', model_name)
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    
    # Generate INT8 static
    int8_path = os.path.join('models', f'int8_{model_name}')
    reader = NpzDataReader('models/samples/calibration_50.npz')
    try:
        quantize_static(model_path, int8_path, reader, quant_format=ort.quantization.QuantFormat.QDQ)
    except Exception as e:
        quantize_dynamic(model_path, int8_path, weight_type=QuantType.QInt8)
        
    candidates = [
        ('onnx_cuda_fp32', model_path, ['CUDAExecutionProvider', 'CPUExecutionProvider'], 'fp32'),
        ('onnx_cuda_int8', int8_path, ['CUDAExecutionProvider', 'CPUExecutionProvider'], 'int8')
    ]
    
    for strat_name, path, providers, prec in candidates:
        print(f'Benchmarking {model_name} [{strat_name}] on NVIDIA T4...')
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess = ort.InferenceSession(path, so, providers=providers)
        
        input_name = sess.get_inputs()[0].name
        x = np.random.randn(1, 3, 224, 224).astype(np.float32)
        
        # Warmup
        for _ in range(15):
            sess.run(None, {input_name: x})
            
        # Latency measurement (50 iterations)
        latencies = []
        for _ in range(50):
            t0 = time.perf_counter()
            sess.run(None, {input_name: x})
            latencies.append((time.perf_counter() - t0) * 1000.0)
            
        p50 = float(np.percentile(latencies, 50))
        p95 = float(np.percentile(latencies, 95))
        fps = 1000.0 / p50 if p50 > 0 else 0
        
        results.append({
            'hardware': 'nvidia_t4_gpu',
            'model': model_name,
            'model_size_mb': os.path.getsize(path)/(1024*1024),
            'strategy': strat_name,
            'framework': 'onnxruntime_gpu',
            'precision': prec,
            'success': True,
            'latency_p50_ms': round(p50, 4),
            'latency_p95_ms': round(p95, 4),
            'throughput_fps': round(fps, 2)
        })

df = pd.DataFrame(results)
df.to_csv('reports/data/benchmark_gpu_t4.csv', index=False)
print('Finished all GPU benchmarks!')
display(df)

# Automatic download to local machine
from google.colab import files
files.download('reports/data/benchmark_gpu_t4.csv')